**RESTAURANTES**


url = https://datos.canarias.es/catalogos/general/dataset/establecimientos-de-hosteleria-y-restauracion-de-tenerife

In [2]:
import pandas as pd
import urllib.parse

df = pd.read_csv(
    r"/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/LLM/tenerife_ocio/dirty_dataset/establecimientos-de-hosteleria-y-restauracion-de-tenerife.csv"
)

def maps_url(nombre, municipio):
    q = urllib.parse.quote(f"{nombre} {municipio} Tenerife")
    return f"https://www.google.com/maps/search/{q}"

rows = []

for municipio, sub in df.groupby("municipio"):

    guach = sub[
        (sub["modalidad"].str.contains("COMERCIALIZACION VINO", case=False, na=False)) &
        (sub["tipo"].str.contains("GUACHINCHE", case=False, na=False))
    ].copy()

    rest = sub[
        (sub["modalidad"].str.contains("RESTAURACION", case=False, na=False)) &
        (sub["tipo"].str.contains("RESTAURANTE", case=False, na=False))
    ].copy()

    if len(guach) > 0:
        g = guach.iloc[0]
        rows.append({
            "municipio": municipio,
            "nombre": g["nombre"],
            "grupo": "Guachinche",
            "url_maps": maps_url(g["nombre"], municipio)
        })

        if len(rest) > 0:
            r = rest.iloc[0]
            rows.append({
                "municipio": municipio,
                "nombre": r["nombre"],
                "grupo": "Restaurante",
                "url_maps": maps_url(r["nombre"], municipio)
            })
        else:
            g2 = guach.iloc[1] if len(guach) > 1 else guach.iloc[0]
            rows.append({
                "municipio": municipio,
                "nombre": g2["nombre"],
                "grupo": "Guachinche",
                "url_maps": maps_url(g2["nombre"], municipio)
            })

    else:
        if len(rest) >= 2:
            r1, r2 = rest.iloc[0], rest.iloc[1]
        elif len(rest) == 1:
            r1 = r2 = rest.iloc[0]
        else:
            continue

        rows.append({
            "municipio": municipio,
            "nombre": r1["nombre"],
            "grupo": "Restaurante",
            "url_maps": maps_url(r1["nombre"], municipio)
        })
        rows.append({
            "municipio": municipio,
            "nombre": r2["nombre"],
            "grupo": "Restaurante",
            "url_maps": maps_url(r2["nombre"], municipio)
        })

restaurantes = pd.DataFrame(rows)
restaurantes = restaurantes.sort_values(["municipio", "grupo", "nombre"])

,municipio,nombre,grupo,url_maps
0,ADEJE,CHEZ MIMI,Restaurante,https://www.google.com/maps/search/CHEZ%20MIMI...
1,ADEJE,LA GRANJA,Restaurante,https://www.google.com/maps/search/LA%20GRANJA...
2,ARAFO,FINCA AGRICOLA KM2KCERO,Guachinche,https://www.google.com/maps/search/FINCA%20AGR...
3,ARAFO,"ROTONDA, LA",Restaurante,https://www.google.com/maps/search/ROTONDA%2C%...
5,ARICO,ANDROMEDA II,Restaurante,https://www.google.com/maps/search/ANDROMEDA%2...
...,...,...,...,...
56,TANQUE,JARDIN DEL NORTE,Restaurante,https://www.google.com/maps/search/JARDIN%20DE...
58,TEGUESTE,MAURO,Restaurante,https://www.google.com/maps/search/MAURO%20TEG...
59,TEGUESTE,MEDEROS,Restaurante,https://www.google.com/maps/search/MEDEROS%20T...
60,VILAFLOR,"MIRADOR, EL",Restaurante,https://www.google.com/maps/search/MIRADOR%2C%...


In [4]:
restaurantes.to_csv(r"C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\datos\LLM\tenerife_ocio\clean_dataset\restaurantes_tenerife.csv", index=False, encoding="utf-8")

**ALOJAMIENTOS**

url = https://datos.canarias.es/catalogos/general/dataset/alojamientos-turisticos-de-tenerife

In [5]:
df = pd.read_csv(r"C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\datos\LLM\tenerife_ocio\dirty_dataset\alojamientos-turisticos-de-tenerife.csv")


def maps_url(nombre, municipio):
    q = urllib.parse.quote(f"{nombre} {municipio} Tenerife")
    return f"https://www.google.com/maps/search/{q}"

# Normalizar texto
df["tipo"] = df["tipo"].astype(str).str.upper()
df["municipio"] = df["municipio"].astype(str).str.strip()

rows = []

extra_rows = [
    {
        "municipio": "Santa Ursula",
        "nombre": "Hotel Spa La Quinta Park Suites",
        "tipo": "HOTEL",
        "categoria": "Hotel"
    },
    {
        "municipio": "Santa Ursula",
        "nombre": "Boutique Hotel Mencey",
        "tipo": "HOTEL",
        "categoria": "Boutique"
    }
]

df_extra = pd.DataFrame(extra_rows)

df = pd.concat([df, df_extra], ignore_index=True)

for municipio, sub in df.groupby("municipio"):

    hoteles = sub[sub["tipo"].str.contains("HOTEL")].copy()
    apts = sub[sub["tipo"].str.contains("APARTAMENTO")].copy()

    otros = sub[~sub["tipo"].str.contains("HOTEL") &
                ~sub["tipo"].str.contains("APARTAMENTO")].copy()

    seleccionados = []
    usados = set()

    if len(hoteles) > 0:
        h1 = hoteles.iloc[0]
        seleccionados.append(("Hotel", h1))
        usados.add(h1["nombre"])

    if len(seleccionados) < 2 and len(apts) > 0:
        a1 = apts.iloc[0]
        if a1["nombre"] not in usados:
            seleccionados.append(("Apartamento", a1))
            usados.add(a1["nombre"])

    if len(seleccionados) < 2:
        for _, h in hoteles.iterrows():
            if h["nombre"] not in usados:
                seleccionados.append(("Hotel", h))
                usados.add(h["nombre"])
                break

    if len(seleccionados) < 2:
        for _, a in apts.iterrows():
            if a["nombre"] not in usados:
                seleccionados.append(("Apartamento", a))
                usados.add(a["nombre"])
                break

    if len(seleccionados) < 2:
        for _, o in otros.iterrows():
            if o["nombre"] not in usados:
                seleccionados.append((o["tipo"].title(), o))
                usados.add(o["nombre"])
                break

    if len(seleccionados) == 1:
        for _, row in sub.iterrows():
            if row["nombre"] not in usados:
                seleccionados.append((row["tipo"].title(), row))
                usados.add(row["nombre"])
                break

    for tipo, row in seleccionados:
        rows.append({
            "municipio": municipio,
            "nombre": row["nombre"],
            "tipo": tipo,
            "url_maps": maps_url(row["nombre"], municipio)
        })

alojamientos = pd.DataFrame(rows)
alojamientos = alojamientos.sort_values(["municipio", "tipo"])
alojamientos

,municipio,nombre,tipo,url_maps
1,ADEJE,PUEBLO TORVISCAS,Apartamento,https://www.google.com/maps/search/PUEBLO%20TO...
0,ADEJE,ROYAL GARDEN RIVER,Hotel,https://www.google.com/maps/search/ROYAL%20GAR...
2,ARAFO,CASA DEL CURA VIEJO,Casa Rural,https://www.google.com/maps/search/CASA%20DEL%...
3,ARAFO,LO DE CARTA,Casa Rural,https://www.google.com/maps/search/LO%20DE%20C...
5,ARICO,"VIÑITA, LA",Casa Emblematica,https://www.google.com/maps/search/VI%C3%91ITA...
4,ARICO,VIÑA VIEJA,Hotel,https://www.google.com/maps/search/VI%C3%91A%2...
7,ARONA,DON JOSE,Apartamento,https://www.google.com/maps/search/DON%20JOSE%...
6,ARONA,MARYLANZA SUITES & SPA,Hotel,https://www.google.com/maps/search/MARYLANZA%2...
9,BUENAVISTA DEL NORTE,CASA QUEMADA,Casa Rural,https://www.google.com/maps/search/CASA%20QUEM...
8,BUENAVISTA DEL NORTE,MELIA HACIENDA DEL CONDE,Hotel,https://www.google.com/maps/search/MELIA%20HAC...


In [6]:
alojamientos.to_csv(r"C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\datos\LLM\tenerife_ocio\clean_dataset\alojamientos_tenerife.csv", index=False, encoding="utf-8")